In [2]:
from pathlib import Path
import tkinter as tk
from tkinter import filedialog
import numpy as np
import pandas as pd

STUDENT_ID = "TUPM-26-4014"
SEED_DIGIT = int(STUDENT_ID[-1])  # 4
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 17
SAMPLES_PER_OUTCOME = 20 + ID_SUM  # 37

Path("output").mkdir(exist_ok=True)

# --- FILE SELECTOR DIALOG ---
print("Please select your 'secom.data' file in the popup window...")
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

data_file = filedialog.askopenfilename(
    title="Select secom.data", filetypes=[("Data files", "*.data"), ("All files", "*.*")]
)
if not data_file:
  raise ValueError("No data file selected!")

print(f"Please select your 'secom_labels.data' file in the popup window...")
labels_file = filedialog.askopenfilename(
    title="Select secom_labels.data",
    filetypes=[("Data files", "*.data"), ("All files", "*.*")],
)
if not labels_file:
  raise ValueError("No labels file selected!")

# Load Data
X_raw = pd.read_csv(data_file, sep=r"\s+", header=None)
y_raw = pd.read_csv(labels_file, sep=r"\s+", header=None)
df = X_raw.copy()
df["Outcome"] = y_raw.iloc[:, 0]
X = df.iloc[:, :-1]

# --- DYNAMIC CALCULATIONS ---
# 1. Class counts and balance sampling
pass_count_orig = int((df["Outcome"] == -1).sum())
fail_count_orig = int((df["Outcome"] == 1).sum())

np.random.seed(SEED_DIGIT)
pass_group = df[df["Outcome"] == -1].sample(
    n=SAMPLES_PER_OUTCOME, random_state=SEED_DIGIT
)
fail_group = df[df["Outcome"] == 1].sample(
    n=SAMPLES_PER_OUTCOME, random_state=SEED_DIGIT
)
balanced_df = pd.concat([pass_group, fail_group]).reset_index(drop=True)

# Imputation for numeric columns
valid_feat_list = list(X.select_dtypes(include=[np.number]).columns[:5])
imputed_df = balanced_df[valid_feat_list + ["Outcome"]].fillna(
    balanced_df[valid_feat_list].mean()
)

# 2. Compute dynamic mean differences between pass (-1) and fail (1) classes
pass_subset = imputed_df[imputed_df["Outcome"] == -1][valid_feat_list]
fail_subset = imputed_df[imputed_df["Outcome"] == 1][valid_feat_list]

mean_diff = fail_subset.mean() - pass_subset.mean()
max_divergence_feature = mean_diff.abs().idxmax()

# 3. Dynamic balance verification check
balanced_pass_count = int((balanced_df["Outcome"] == -1).sum())
balanced_fail_count = int((balanced_df["Outcome"] == 1).sum())
balance_verified = bool(
    balanced_pass_count == balanced_fail_count
    == SAMPLES_PER_OUTCOME
)

# Export results
balanced_df.to_csv("output/balanced_sample.csv", index=False)
mean_diff.to_frame("Mean_Difference").to_csv("output/feature_comparison.csv")

print("\n--- Exercise 2 Assessment Data (Dynamic) ---")
print(f"Original Pass Count (-1): {pass_count_orig}")
print(f"Original Fail Count (1): {fail_count_orig}")
print(f"Balanced Samples Per Outcome: {SAMPLES_PER_OUTCOME}")
print(f"Feature with Maximum Divergence: Feature {max_divergence_feature}")
print(f"Class Balance Verification Passed: {balance_verified}")

Please select your 'secom.data' file in the popup window...
Please select your 'secom_labels.data' file in the popup window...

--- Exercise 2 Assessment Data (Dynamic) ---
Original Pass Count (-1): 1463
Original Fail Count (1): 104
Balanced Samples Per Outcome: 37
Feature with Maximum Divergence: Feature 3
Class Balance Verification Passed: True
